# Predicción del consumo eléctrico de una vivienda: LSTM frente a SARIMA

**Asignatura:** Redes Neuronales y Deep Learning — Maestría en Estadística con mención en Ciencia de Datos e Inteligencia Artificial (ESPOCH)
**Grupo 2:** Carlos Cuenca, Wilson Sani, William Allauca, Gustavo Chaglla

**Pregunta:** ¿una red neuronal recurrente LSTM pronostica el consumo horario mejor que un modelo SARIMA?

**Datos:** *Individual Household Electric Power Consumption* (Hébrail y Bérard, UCI Machine Learning Repository, licencia CC BY 4.0).

**Cómo usarlo:** `Entorno de ejecución → Ejecutar todo`. Al final se descarga `resultados.zip` con los CSV y los gráficos que usan el dashboard y el portal web.

**Limitaciones de esta versión (se reportan en el informe):**
1. Se usan los últimos 24 meses de la serie, no los 47 disponibles.
2. La LSTM se entrena una sola vez (una configuración, una semilla), sin ajuste de hiperparámetros.
3. SARIMA tiene orden fijo, estacionalidad diaria (s = 24) y se ajusta con los últimos 90 días de entrenamiento; no modela la estacionalidad semanal.
4. Los datos provienen de una sola vivienda: las conclusiones no se generalizan a otros hogares.

## 1. Librerías y parámetros

In [ ]:
import os, zipfile, random, time, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 2026
random.seed(SEMILLA); np.random.seed(SEMILLA)
import tensorflow as tf
tf.keras.utils.set_random_seed(SEMILLA)
from statsmodels.tsa.statespace.sarimax import SARIMAX

# --- Parámetros del estudio ---
MESES_SERIE   = 24       # meses de la serie que se usan
DIAS_VAL      = 92       # ~3 meses de validación
DIAS_PRUEBA   = 92       # ~3 meses de prueba
VENTANA       = 48       # horas anteriores que ve la LSTM
UNIDADES      = 32       # neuronas de la capa LSTM
EPOCAS_MAX    = 60
PACIENCIA     = 5
LOTE          = 64
ORDEN         = (1, 0, 1)          # SARIMA (p, d, q)
ORDEN_EST     = (0, 1, 1, 24)      # SARIMA (P, D, Q, s)
DIAS_SARIMA   = 90       # días finales de entrenamiento con que se ajusta SARIMA

os.makedirs('resultados', exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('TensorFlow', tf.__version__)

## 2. Descarga y lectura de los datos

In [ ]:
URL = 'https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip'
TXT = 'household_power_consumption.txt'

if not os.path.exists(TXT):
    try:
        urllib.request.urlretrieve(URL, 'datos.zip')
    except Exception as e:
        # Si la descarga falla, subir manualmente el .zip descargado de UCI
        print('No se pudo descargar:', e, '\nSuba el archivo .zip de UCI:')
        from google.colab import files
        subido = files.upload()
        os.rename(list(subido)[0], 'datos.zip')
    with zipfile.ZipFile('datos.zip') as z:
        z.extractall('.')

df = pd.read_csv(TXT, sep=';', na_values=['?'],
                 usecols=['Date', 'Time', 'Global_active_power'],
                 dtype={'Global_active_power': 'float64'})
df['fecha_hora'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
print(f'Mediciones por minuto: {len(df):,}')
print(f'Periodo: {df.fecha_hora.min()}  a  {df.fecha_hora.max()}')
print(f'Minutos sin dato: {df.Global_active_power.isna().sum():,} '
      f'({100 * df.Global_active_power.isna().mean():.2f} %)')

## 3. Serie horaria de energía (kWh)

La potencia activa viene en kW promediada por minuto. La energía de una hora es el promedio de la potencia de esa hora multiplicado por 1 h, es decir, kWh.
Una hora con menos de 45 minutos válidos se considera faltante y se imputa con el valor de **la misma hora de la semana anterior** (solo usa información pasada).

In [ ]:
potencia = df.set_index('fecha_hora')['Global_active_power']
grupo = potencia.resample('h')
kwh = grupo.mean()
kwh[grupo.count() < 45] = np.nan

# Índice horario completo y recorte a días completos
kwh = kwh.reindex(pd.date_range(kwh.index.min(), kwh.index.max(), freq='h'))
dia = kwh.index.normalize()
kwh = kwh[(dia > dia.min()) & (dia < dia.max())]

faltante = kwh.isna()
while kwh.isna().any():
    antes = kwh.isna().sum()
    kwh = kwh.fillna(kwh.shift(168))
    if kwh.isna().sum() == antes:
        break
kwh = kwh.ffill().bfill()

# Últimos MESES_SERIE meses
inicio = kwh.index.max() - pd.DateOffset(months=MESES_SERIE) + pd.Timedelta(hours=1)
serie = kwh[inicio:]
imputado = faltante[inicio:]
print(f'Serie horaria usada: {len(serie):,} horas, de {serie.index.min()} a {serie.index.max()}')
print(f'Horas imputadas: {int(imputado.sum())} ({100 * imputado.mean():.2f} %)')

pd.DataFrame({'fecha_hora': serie.index, 'kwh': serie.round(4).values,
              'imputado': imputado.astype(int).values}).to_csv('resultados/serie_horaria.csv', index=False)

## 4. Partición cronológica

In [ ]:
n = len(serie)
n_prueba = DIAS_PRUEBA * 24
n_val = DIAS_VAL * 24
n_ent = n - n_val - n_prueba

ent    = serie.iloc[:n_ent]
val    = serie.iloc[n_ent:n_ent + n_val]
prueba = serie.iloc[n_ent + n_val:]

particion = pd.DataFrame({
    'conjunto': ['Entrenamiento', 'Validación', 'Prueba'],
    'desde': [ent.index.min(), val.index.min(), prueba.index.min()],
    'hasta': [ent.index.max(), val.index.max(), prueba.index.max()],
    'horas': [len(ent), len(val), len(prueba)]})
particion.to_csv('resultados/particion.csv', index=False)
particion

## 5. Análisis exploratorio

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 9))
diaria = serie.resample('D').sum()
ax[0].plot(diaria.index, diaria.values, lw=0.8, color='#1d4ed8')
ax[0].axvspan(val.index.min(), val.index.max(), color='orange', alpha=0.15, label='Validación')
ax[0].axvspan(prueba.index.min(), prueba.index.max(), color='red', alpha=0.15, label='Prueba')
ax[0].set_title('Consumo diario (kWh)'); ax[0].set_ylabel('kWh/día'); ax[0].legend()

perfil_hora = ent.groupby(ent.index.hour).mean()
ax[1].bar(perfil_hora.index, perfil_hora.values, color='#0284c7')
ax[1].set_title('Patrón diario: consumo medio por hora (entrenamiento)')
ax[1].set_xlabel('Hora del día'); ax[1].set_ylabel('kWh')

dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
perfil_dia = ent.groupby(ent.index.dayofweek).mean()
ax[2].bar(dias, perfil_dia.values, color='#0284c7')
ax[2].set_title('Patrón semanal: consumo medio horario por día (entrenamiento)'); ax[2].set_ylabel('kWh')
plt.tight_layout(); plt.savefig('resultados/fig1_serie_patrones.png', dpi=150); plt.show()

perfil_hora.rename('kwh').rename_axis('hora').round(4).to_csv('resultados/perfil_hora.csv')
pd.DataFrame({'dia': dias, 'kwh': perfil_dia.round(4).values}).to_csv('resultados/perfil_dia.csv', index=False)

## 6. Métricas

- $\text{MAE} = \frac{1}{n}\sum |y_t - \hat y_t|$
- $\text{RMSE} = \sqrt{\frac{1}{n}\sum (y_t - \hat y_t)^2}$
- $\text{MASE} = \dfrac{\text{MAE}}{\text{MAE del pronóstico estacional (24 h) en entrenamiento}}$. Un valor menor que 1 indica que el modelo supera a esa referencia.

In [ ]:
escala_mase = np.mean(np.abs(ent.values[24:] - ent.values[:-24]))

def metricas(real, pred):
    e = np.asarray(real) - np.asarray(pred)
    mae = np.mean(np.abs(e))
    return {'MAE': mae, 'RMSE': np.sqrt(np.mean(e ** 2)), 'MASE': mae / escala_mase}

print(f'Escala del MASE (MAE estacional en entrenamiento): {escala_mase:.4f} kWh')

## 7. Modelos de referencia

- **Estacional:** el consumo de la misma hora del día anterior, $\hat y_t = y_{t-24}$.
- **Persistencia:** el consumo de la hora anterior, $\hat y_t = y_{t-1}$. Se añade porque, a una hora vista, suele ser la referencia más difícil de superar.

In [ ]:
pred = pd.DataFrame({'real': prueba})
pred['estacional'] = serie.shift(24)[prueba.index]
pred['persistencia'] = serie.shift(1)[prueba.index]

## 8. SARIMA

Modelo SARIMA$(1,0,1)(0,1,1)_{24}$. Se estiman los parámetros **una sola vez** con datos de entrenamiento. Después se aplica el filtro con esos parámetros fijos sobre el resto de la serie, de modo que cada pronóstico de prueba usa los datos reales hasta la hora anterior, igual que la LSTM, pero sin reestimar nada con datos de prueba.

In [ ]:
t0 = time.time()
ent_sarima = ent.iloc[-DIAS_SARIMA * 24:]
ajuste = SARIMAX(ent_sarima, order=ORDEN, seasonal_order=ORDEN_EST).fit(disp=False)
print(f'Ajuste en {time.time() - t0:.0f} s  |  AIC = {ajuste.aic:.1f}')
print(ajuste.params.round(4).to_string())

# Pronósticos a una hora vista con parámetros fijos
completa = serie[ent_sarima.index.min():]
filtro = SARIMAX(completa, order=ORDEN, seasonal_order=ORDEN_EST).filter(ajuste.params)
pred['sarima'] = filtro.predict(start=prueba.index.min(), end=prueba.index.max())

ajuste.params.round(5).rename('valor').rename_axis('parametro').to_csv('resultados/sarima_parametros.csv')

## 9. Red LSTM

- **Entrada:** las 48 horas anteriores (escaladas). **Salida:** el consumo de la hora siguiente.
- **Arquitectura:** LSTM(32) → Dense(1). Optimizador Adam, pérdida MSE.
- **Escalado:** media y desviación calculadas solo con entrenamiento.
- **Early Stopping** sobre `val_loss` con paciencia 5 y restauración de los mejores pesos. El conjunto de prueba no interviene en ninguna decisión.

In [ ]:
media, desv = ent.mean(), ent.std()
z = ((serie - media) / desv).values.astype('float32')

def ventanas(ini, fin):
    X = np.stack([z[i - VENTANA:i] for i in range(ini, fin)])[..., None]
    return X, z[ini:fin]

X_ent, y_ent = ventanas(VENTANA, n_ent)
X_val, y_val = ventanas(n_ent, n_ent + n_val)
X_pru, y_pru = ventanas(n_ent + n_val, n)
print('Entrenamiento:', X_ent.shape, ' Validación:', X_val.shape, ' Prueba:', X_pru.shape)

modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(VENTANA, 1)),
    tf.keras.layers.LSTM(UNIDADES),
    tf.keras.layers.Dense(1)])
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
modelo.summary()

t0 = time.time()
parada = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True)
historia = modelo.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=EPOCAS_MAX,
                      batch_size=LOTE, callbacks=[parada], verbose=2)
print(f'Entrenamiento en {time.time() - t0:.0f} s')

pred['lstm'] = modelo.predict(X_pru, verbose=0).ravel() * desv + media

In [ ]:
h = pd.DataFrame(historia.history)
h.insert(0, 'epoca', range(1, len(h) + 1))
h.round(5).to_csv('resultados/historial_lstm.csv', index=False)
mejor = int(h.val_loss.idxmin()) + 1

plt.figure(figsize=(8, 4.5))
plt.plot(h.epoca, h.loss, 'o-', label='Entrenamiento (loss)', color='#dc2626')
plt.plot(h.epoca, h.val_loss, 'o-', label='Validación (val_loss)', color='#0891b2')
plt.axvline(mejor, ls='--', color='gray', label=f'Mejor época = {mejor}')
plt.xlabel('Época'); plt.ylabel('MSE (datos escalados)'); plt.title('Curvas de entrenamiento de la LSTM'); plt.legend()
plt.tight_layout(); plt.savefig('resultados/fig2_curvas_lstm.png', dpi=150); plt.show()
print(f'Épocas ejecutadas: {len(h)}  |  mejor época: {mejor}  |  parámetros: {modelo.count_params():,}')

## 10. Comparación en el conjunto de prueba

In [ ]:
nombres = {'lstm': 'LSTM', 'sarima': 'SARIMA', 'estacional': 'Estacional (24 h)', 'persistencia': 'Persistencia (1 h)'}
tabla = pd.DataFrame({nombres[k]: metricas(pred.real, pred[k]) for k in nombres}).T
tabla = tabla.sort_values('MAE')
tabla.index.name = 'modelo'
tabla.round(4).to_csv('resultados/metricas.csv')

salida = pred.round(4).copy()
salida.insert(0, 'fecha_hora', salida.index)
salida.to_csv('resultados/predicciones.csv', index=False)
tabla.round(4)

In [ ]:
colores = {'lstm': '#dc2626', 'sarima': '#16a34a', 'estacional': '#9ca3af', 'persistencia': '#f59e0b'}

# Predicciones frente a valores reales: primera semana de prueba
sem = pred.iloc[:7 * 24]
plt.figure(figsize=(12, 4.8))
plt.plot(sem.index, sem.real, color='black', lw=1.8, label='Real')
for k in ['lstm', 'sarima', 'estacional']:
    plt.plot(sem.index, sem[k], color=colores[k], lw=1.1, label=nombres[k])
plt.ylabel('kWh'); plt.title('Pronóstico a una hora vista: primera semana del periodo de prueba'); plt.legend(ncol=4)
plt.tight_layout(); plt.savefig('resultados/fig3_pred_vs_real.png', dpi=150); plt.show()

# ¿Cuándo se equivocan más? Error absoluto medio por hora del día
err = pred[list(nombres)].sub(pred.real, axis=0).abs()
err_hora = err.groupby(err.index.hour).mean()
plt.figure(figsize=(11, 4.5))
for k in nombres:
    plt.plot(err_hora.index, err_hora[k], 'o-', color=colores[k], label=nombres[k], ms=4)
plt.xlabel('Hora del día'); plt.ylabel('MAE (kWh)'); plt.xticks(range(0, 24))
plt.title('Error absoluto medio por hora del día (prueba)'); plt.legend()
plt.tight_layout(); plt.savefig('resultados/fig4_error_por_hora.png', dpi=150); plt.show()
err_hora.rename_axis('hora').round(4).to_csv('resultados/error_por_hora.csv')

# Comparación de métricas
ax = tabla[['MAE', 'RMSE']].plot.bar(figsize=(8, 4.5), color=['#1d4ed8', '#0284c7'], rot=0)
ax.set_ylabel('kWh'); ax.set_title('MAE y RMSE en prueba'); ax.set_xlabel('')
plt.tight_layout(); plt.savefig('resultados/fig5_metricas.png', dpi=150); plt.show()

## 11. Respuesta a la pregunta

In [ ]:
m = tabla['MAE']
dif = 100 * (m['SARIMA'] - m['LSTM']) / m['SARIMA']
print(f"MAE en prueba  ->  LSTM: {m['LSTM']:.4f} kWh   SARIMA: {m['SARIMA']:.4f} kWh")
if dif > 0:
    print(f'La LSTM tuvo un MAE {dif:.1f} % menor que SARIMA.')
else:
    print(f'SARIMA tuvo un MAE {-dif:.1f} % menor que la LSTM.')
print(f'Mejor modelo por MAE: {tabla.index[0]}')
print('\nNota: la diferencia proviene de un solo entrenamiento de la LSTM y de una sola vivienda;'
      '\nindica el resultado de este caso, no una superioridad general de un método.')

## 12. Descarga de resultados

In [ ]:
with zipfile.ZipFile('resultados.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir('resultados')):
        z.write(os.path.join('resultados', f), f)
print(sorted(os.listdir('resultados')))
try:
    from google.colab import files
    files.download('resultados.zip')
except Exception:
    print('resultados.zip creado en la carpeta de trabajo')